In [ ]:
%reload_ext autoreload
%autoreload 2

# Agent-Based Task Execution

This notebook demonstrates how to use the `Agent` pipeline from **[OnPrem.LLM](https://github.com/amaiya/onprem)** to create autonomous agents that can execute complex tasks using a variety of tools.

## The AgentExecutor

The `AgentExecutor` allows you to launch AI agents to solve various tasks using both cloud and local models. We will use **gpt-5-mini** (cloud) and **glm-4.7-flash** (local) for these examples.

By default, the `AgentExecutor` has access to 9 built-in tools.  You remove access to built-in-tools as necessary.  You can optionally give the agent access to [custom tools](https://amaiya.github.io/patchpal/features/custom-tools/) by placing them in `~/.patchpal/tools`.

The `AgentExecutor` is implemented using our coding agent, PatchPal, which you'll need to install: `pip install patchpal`.

In [ ]:
# | notest

from onprem.pipelines import AgentExecutor

In [ ]:
# | notest

AgentExecutor.print_default_tools()

AgentExecutor Default Tools

These tools are used by default when enabled_tools=None:

   1. read_file       - Read complete file contents
   2. read_lines      - Read specific line ranges from files
   3. edit_file       - Edit files via find/replace
   4. write_file      - Write complete file contents
   5. grep            - Search for patterns in files
   6. find            - Find files by glob pattern
   7. run_shell       - Execute shell commands
   8. web_search      - Search the web for information
   9. web_fetch       - Fetch and read content from URLs

Customization Examples:

# Use defaults (all tools including shell):
executor = AgentExecutor(model='anthropic/claude-sonnet-4-5')

# Defaults but no shell access (safer):
executor = AgentExecutor(
    model='openai/gpt-5-mini',
    disable_shell=True
)

# Minimal tools:
executor = AgentExecutor(
    model='openai/gpt-5-mini',
    enabled_tools=['read_file', 'write_file']
)

# Web research only:
executor = AgentExecutor(
    mo

## Examples

Let's run through some examples for different scenarios.

### Basic Calculator Example

In this introductory example, we will launch an agent to build a **calculator module in Python**.

By default, the agent will operate within the working_directory you specify (or the current folder if not working directory is specified).  The agent cannot read or write outside the working directory.

**Important Note:**:  If the agent has access to the `run_shell` tool, it can potentially read or modify files outside of the working directory (e.g., auto-generating and executing a Python script that writes files outside of working directory).  For these reasons, you can either supply the `disable_shell=True` to remove shell access or `sandbox=True`, which runs the agent in an ephemeral container.  We will cover sandboxed execution later. 

We will run this example without either, as we are running this notebook within a VM.

In [ ]:
# | notest

executor = AgentExecutor(
     model='openai/gpt-5-mini', # assumes OPENAI_API_KEY is already set as environment variable
     max_iterations=10
 )

result = executor.run(
     task="""
     Create a simple Python calculator module with the following:
     - calculator.py with add, subtract, multiply, divide functions
     - test_calculator.py with pytest tests
     - All tests must pass
     """,
     working_dir='./calculator_project'
)
print(result)


✈️  PatchPal Autopilot Mode Starting
Prompt: 
     Create a simple Python calculator module with the following:
     - calculator.py with add, su...
Completion promise: 'COMPLETE'
Max iterations: 10
Model: openai/gpt-5-mini
Working directory: /home/amaiya/projects/ghub/onprem/nbs/calculator_project
🔒 File access restricted to working directory


🔄 Autopilot Iteration 1/10

🤔 Thinking...

I'll inspect the repository to see what's already present, then add the         
calculator module and tests. I'll run the test suite and fix any issues until   
all tests pass. I'll show test run output and then finish with the required     
completion tag. I'll list files first.                                          

📂 Finding files: **/*
🤔 Thinking...
📖 Reading: task_prompt.md
🤔 Thinking...

I'll create calculator.py and test_calculator.py, then run the pytest suite to  
confirm all tests pass. I'll write the files now.                               

📝 Patching: calculator.py
🤔 Thinking...
📝 P

### Web Research Agent

This example is a web research agent that only has access to the following tools: `web_search`, `web_fetch`, `write_file`.

In [ ]:
# | notest

prompt = """
Research the latest developments in quantum computing in 2026.
Create a markdown report called 'quantum_computing_2026.md' with:
- Executive summary
- Key breakthroughs
- Major companies/institutions involved
- Potential applications
- Sources and references
"""

In [ ]:
# | notest

executor = AgentExecutor(
     model='openai/gpt-5-mini',
     max_iterations=10,
    enabled_tools=["web_search", "web_fetch", "write_file"],
 )

result = executor.run(
     task=prompt,
     working_dir='./quantum_report'
)


✈️  PatchPal Autopilot Mode Starting
Prompt: 
Research the latest developments in quantum computing in 2026.
Create a markdown report called 'qua...
Completion promise: 'COMPLETE'
Max iterations: 10
Model: openai/gpt-5-mini
Working directory: /home/amaiya/projects/ghub/onprem/nbs/quantum_report
🔒 File access restricted to working directory


🔄 Autopilot Iteration 1/10

🤔 Thinking...

I'll search the web for recent (2024–2026) developments, company updates, and   
major milestones in quantum computing to build the report, then create the      
markdown file. I'll run several targeted searches (hardware, QEC, companies, and
2026-specific news) and collect key sources.                                    

🌐 Searching web: quantum computing 2026 developments 2026 breakthrough 2025 2026 news
Impersonate 'chrome_124' does not exist, using 'random'
🤔 Thinking...
🌐 Searching web: Google Quantum 2026 breakthrough Quantum Echoes 2026 'Quantum Echoes' algorithm Google 2026
🤔 Thinking...
🌐 Search

## Local Models

The `AgentExecutor` supports local models.  By default, it will assume the local model supports native function-calling (e.g., gpt-oss-120b).  If you use a local model that does **not** have good native support for function-calling (a.k.a. tool-calling), you can change the agent_type to `react`. In this example, we will use `llama3.1:8b`.

**Note:** The default context window length in Ollama is typically too small for agentic  workflows.  Depending on the model and task, we recommend inreasing to at least 8192.  Reasoning models like gpt-oss:120b may require 32K or 64K.  

```sh
OLLAMA_CONTEXT_LENGTH=32000 ollama serve
```

In [ ]:
# | notest

executor = AgentExecutor(
     model='ollama_chat/glm-4.7-flash:q4_K_M',
     enabled_tools=["web_fetch", "write_file"],
     max_iterations=10
 )

result = executor.run(
     task="What is the highest level of education of the person listed on this page: https://arun.maiya.net? Write answer in answer.txt.",
     working_dir='./extraction_example'
)


✈️  PatchPal Autopilot Mode Starting
Prompt: What is the highest level of education of the person listed on this page: https://arun.maiya.net? Wr...
Completion promise: 'COMPLETE'
Max iterations: 10
Model: ollama_chat/glm-4.7-flash:q4_K_M
Working directory: /home/amaiya/projects/ghub/onprem/nbs/extraction_example
🔒 File access restricted to working directory


🔄 Autopilot Iteration 1/10

🤔 Thinking...

I'll fetch the webpage to find the highest level of education listed for the    
person.                                                                         

🌐 Fetching: https://arun.maiya.net
🤔 Thinking...

I found the education information on the page. The person states: "I completed a
Ph.D. in Computer Science at the Laboratory for Computational Population        
Biology..." The highest level of education is a Ph.D.                           

I'll write this to answer.txt.                                                  

📝 Patching: answer.txt
🤔 Thinking...

📝 Agent Response

In [ ]:
!ls ./extraction_example/

answer.txt


In [ ]:
!cat ./extraction_example/answer.txt

Ph.D. in Computer Science

## Sandboxed Execution

For enhanced security and isolation, set sandbox=True to run the agent in an ephemeral Docker/Podman container. This is useful when working with
untrusted code, needing resource limits, or wanting to protect your file system from accidental modifications. The tradeoff is ~5-10 seconds of
container startup overhead, but you gain a clean, reproducible environment that's automatically cleaned up after execution.

> Prerequisites: Requires Docker or Podman installed. See docker.com or podman.io. Mac/Windows Podman users: run podman machine init && podman
▌ machine start first.


In [ ]:
# | notest

prompt = """
Create a Python script that:
1. Generates sample sales data for 12 months (random)
2. Calculates total sales, average, min, max
3. Creates a matplotlib bar chart saved as 'sales_chart.png'
4. Writes a summary report to 'sales_analysis.txt'
"""

In [ ]:
# | notest

executor = AgentExecutor(
     model='openai/gpt-5-mini',
     max_iterations=10,
    sandbox=True
 )

result = executor.run(
     task=prompt,
     working_dir='./data_analysis'
)

Unable to find image 'python:3.11-slim' locally
3.11-slim: Pulling from library/python
206356c42440: Pulling fs layer
206709444425: Pulling fs layer
bbf418262cbe: Pulling fs layer
43272e9c9d4b: Pulling fs layer
43272e9c9d4b: Waiting
206709444425: Verifying Checksum
206709444425: Download complete
43272e9c9d4b: Download complete
bbf418262cbe: Verifying Checksum
bbf418262cbe: Download complete
206356c42440: Download complete
206356c42440: Pull complete
206709444425: Pull complete
bbf418262cbe: Pull complete
43272e9c9d4b: Pull complete
Digest: sha256:d6e4d224f70f9e0172a06a3a2eba2f768eb146811a349278b38fff3a36463b47
Status: Downloaded newer image for python:3.11-slim

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

✈️  PatchPal Autopilot Mode Starting
Prompt: 
Create a Python script that:
1. Generates sample sales data for 12 months (random)
2. Calculates to...
Completion promise: 'COMPLETE'
Max iterations: 10
Model: openai/gpt-

In [ ]:
!ls ./data_analysis

generate_sales_report.py


In [ ]:
!cat ./data_analysis/generate_sales_report.py

#!/usr/bin/env python3
"""
Generate sample sales data for 12 months, compute stats, save chart and summary.
Creates:
 - sales_chart.png
 - sales_analysis.txt
"""
import random
import statistics
import os
import matplotlib
matplotlib.use('Agg')  # use non-interactive backend suitable for scripts
import matplotlib.pyplot as plt


def generate_sales(months=12, low=5000, high=20000):
    """Return a list of `months` random sales values rounded to 2 decimals."""
    return [round(random.uniform(low, high), 2) for _ in range(months)]


def analyze(sales, months_names):
    total = round(sum(sales), 2)
    avg = round(statistics.mean(sales), 2)
    minimum = min(sales)
    maximum = max(sales)
    min_month = months_names[sales.index(minimum)]
    max_month = months_names[sales.index(maximum)]
    return dict(total=total, average=avg, min=minimum, max=maximum, min_month=min_month, max_month=max_month)


def save_chart(months_names, sales, filename='sales_chart.png'):
    fig, ax = plt.subplot

 ## Local Models + Sandbox: Networking Setup

 Local models (Ollama, llama.cpp) on localhost need container networking configured:

 - **Linux/WSL2**: Supply `network='host'` to `AgentExecutor`. (WSL2: make sure to enable mirrored networking in `.wslconfig`.)
 - **macOS/Windows**: Set `OLLAMA_API_BASE='http://host.docker.internal:11434'` (Docker) or `http://host.containers.internal:11434` (Podman)